# Neural Local Volatility v6 — ~90% complete benchmark

This version is deliberately a **closure build**, not another notebook expansion. It keeps the CPU-friendly Sabaté-style control variate training from v5, but adds the missing correctness and generalisation checks needed before moving to Neural SLV.

**Main run (execute first):** 80/20 real-SPX held-out calibration, vega/maturity-balanced loss, v5 warm-start when available, corrected spot delta, fixed-clock next-day pricing, and one-day hedge audit.

**Optional experiments (run separately, not as “Run All”):**
1. shared network vs shared-trunk + maturity-head architecture;
2. reduced Gierjatowicz/Sabaté synthetic Heston replication.

The final multi-date pricing/hedging study and three-seed robustness check are intentionally left for v7.


## 1. Setup, CPU budget and experiment switches


In [21]:
from pathlib import Path
from dataclasses import dataclass
import copy, gc, json, math, os, random, time, warnings

# Conservative native-threading limits: v5 had native-kernel crashes on this laptop.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.integrate import quad
from scipy.optimize import brentq
from scipy.special import ndtr
from tqdm import tqdm
import torch
import torch.nn as nn

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 12345
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch, "xpu"):
        try:
            if torch.xpu.is_available():
                return torch.device("xpu")
        except Exception:
            pass
    return torch.device("cpu")

DEVICE = choose_device()
if DEVICE.type == "cpu":
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass

# ------------------------- main v6 choices -------------------------
EXAMPLE_DATE = "2024-03-01"
MODEL_ARCH = "shared"              # main first; optional ablation tests "maturity_heads"
WARM_START_FROM_V5 = True
FORCE_RETRAIN = True

HELDOUT_FRACTION = 0.20
VEGA_FLOOR_QUANTILE = 0.10
MAX_QUOTES_PER_SLICE = 25

# Same basic CPU budget as v5, but a v5 warm-start uses fewer epochs.
COLD_START_EPOCHS = 1_000
WARM_START_EPOCHS = 600
PATHS_PER_STEP = 4_096
STEPS_PER_EPOCH = 10
QUOTES_PER_MATURITY_PER_STEP = 12
FIXED_RAW_EVAL_PATHS = 16_384
EVAL_EVERY_SDE_EPOCHS = 10
TOP_K_CHECKPOINTS = 5
RERANK_PATHS = 50_000
FINAL_MC_PATHS = 100_000
FINAL_PATH_CHUNK = 10_000
RESUME_SAVE_EVERY_EPOCHS = 10

# Sabaté-style network / optimizer constants.
SDE_HIDDEN_LAYERS = 4
SDE_WIDTH = 50
HEDGE_HIDDEN_LAYERS = 3
HEDGE_WIDTH = 30
LEARNING_RATE = 1e-3
SDE_GRAD_CLIP = 5.0
HEDGE_GRAD_CLIP = 3.0
BASE_STEPS_PER_YEAR = 96

# Delta audit: contractual K fixed; forwards scale with the spot bump.
DELTA_BUMP_BPS = (1, 5, 10)
DELTA_HEADLINE_BPS = 5

# Optional experiments are OFF by default. Run them after the main result.
RUN_MATURITY_HEAD_ABLATION = True
RUN_SYNTHETIC_REPLICATION = True

CACHE_DIR = Path("neural_lv_v6_cache")
CACHE_DIR.mkdir(exist_ok=True)
V5_CACHE_DIR = Path("neural_lv_v5_cache")

print("Device:", DEVICE, "| PyTorch threads:", torch.get_num_threads())
print("Main architecture:", MODEL_ARCH)
print("Main design: 80/20 held-out, vega-normalised maturity-balanced loss")
print(f"Paths/step={PATHS_PER_STEP:,}, steps/epoch={STEPS_PER_EPOCH}, final MC={FINAL_MC_PATHS:,}")
print("Optional architecture/synthetic experiments are disabled by default.")


Device: cpu | PyTorch threads: 1
Main architecture: shared
Main design: 80/20 held-out, vega-normalised maturity-balanced loss
Paths/step=4,096, steps/epoch=10, final MC=100,000
Optional architecture/synthetic experiments are disabled by default.


## 2. Load and clean the OptionMetrics data


In [4]:
def find_file(pattern):
    search_dirs = [Path("."), Path("/mnt/data")]
    found = []
    for folder in search_dirs:
        found.extend(sorted(folder.glob(pattern)))
    if not found:
        raise FileNotFoundError(f"Could not find a file matching {pattern!r}")
    return found[0]


FILES = {
    "options": find_file("one_month_spx_option_price.csv"),
    "forwards": find_file("one_month_spx_forward_price.csv"),
    "spot": find_file("one_month_spx_security_price.csv"),
    "rates": find_file("one_month_spx_zero_coupon_yield_curve.csv"),
}

for name, path in FILES.items():
    print(f"{name:8s}: {path}")

options_raw = pd.read_csv(FILES["options"], low_memory=False)
forwards_raw = pd.read_csv(FILES["forwards"], low_memory=False)
spot_raw = pd.read_csv(FILES["spot"], low_memory=False)
rates_raw = pd.read_csv(FILES["rates"], low_memory=False)



SECID = 108105
MIN_DTE, MAX_DTE = 7, 365
MAX_ABS_LOG_MONEYNESS = 0.30
MIN_ABS_DELTA, MAX_ABS_DELTA = 0.05, 0.95
MAX_RELATIVE_SPREAD = 0.25
ATM_TIGHTER_SIDE_BAND = 0.05
MIN_POINTS_PER_SLICE = 8


def prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw):
    options = options_raw.copy()
    options["date"] = pd.to_datetime(options["date"])
    options["exdate"] = pd.to_datetime(options["exdate"])
    options["K"] = options["strike_price"] / 1000.0
    options["T_days"] = (options["exdate"] - options["date"]).dt.days
    options["T"] = options["T_days"] / 365.0
    options["bid"] = pd.to_numeric(options["best_bid"], errors="coerce")
    options["ask"] = pd.to_numeric(options["best_offer"], errors="coerce")
    options["mid"] = 0.5 * (options["bid"] + options["ask"])
    options["spread"] = options["ask"] - options["bid"]
    options["relative_spread"] = options["spread"] / options["mid"].replace(0, np.nan)
    options["iv_market"] = pd.to_numeric(options["impl_volatility"], errors="coerce")
    options["delta_market"] = pd.to_numeric(options["delta"], errors="coerce")
    options["abs_delta_market"] = options["delta_market"].abs()

    mask = (
        (options["secid"] == SECID)
        & (options["exercise_style"] == "E")
        & (options["contract_size"] == 100)
        & (options["cfadj"] == 1)
        & options["cp_flag"].isin(["C", "P"])
        & (options["bid"] >= 0)
        & (options["ask"] > options["bid"])
        & (options["mid"] > 0)
        & options["T_days"].between(MIN_DTE, MAX_DTE)
        & options["iv_market"].between(1e-4, 5.0)
        & options["abs_delta_market"].between(MIN_ABS_DELTA, MAX_ABS_DELTA)
        & (options["relative_spread"] <= MAX_RELATIVE_SPREAD)
    )
    options = options.loc[mask].copy()

    spot = spot_raw.loc[spot_raw["secid"] == SECID, ["date", "close"]].copy()
    spot["date"] = pd.to_datetime(spot["date"])
    spot = spot.rename(columns={"close": "S"}).drop_duplicates("date")

    forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
    forwards["date"] = pd.to_datetime(forwards["date"])
    forwards["expiration"] = pd.to_datetime(forwards["expiration"])
    forwards = forwards.rename(columns={
        "expiration": "exdate",
        "AMSettlement": "am_settlement",
        "ForwardPrice": "F",
    })
    forwards = forwards[
        ["date", "exdate", "am_settlement", "F"]
    ].drop_duplicates(["date", "exdate", "am_settlement"])

    rates = rates_raw.copy()
    rates["date"] = pd.to_datetime(rates["date"])
    rates["days"] = pd.to_numeric(rates["days"], errors="coerce")
    rates["rate_decimal"] = pd.to_numeric(rates["rate"], errors="coerce") / 100.0
    rate_curves = {
        d: (g["days"].to_numpy(float), g["rate_decimal"].to_numpy(float))
        for d, g in rates.dropna().sort_values(["date", "days"]).groupby("date")
    }

    options = options.merge(spot, on="date", how="left").merge(
        forwards,
        on=["date", "exdate", "am_settlement"],
        how="left",
        validate="many_to_one",
    )
    options["r"] = [
        float(np.interp(n, *rate_curves[d])) if d in rate_curves else np.nan
        for d, n in zip(options["date"], options["T_days"])
    ]
    options["D"] = np.exp(-options["r"] * options["T"])
    options["carry"] = np.log(options["F"] / options["S"]) / options["T"]
    options["q"] = options["r"] - options["carry"]
    options["k"] = np.log(options["K"] / options["F"])

    options = options.dropna(subset=["S", "F", "r", "D", "carry", "k"])
    options = options.loc[options["k"].abs() <= MAX_ABS_LOG_MONEYNESS].copy()

    call_lb = options["D"] * np.maximum(options["F"] - options["K"], 0.0)
    call_ub = options["D"] * options["F"]
    put_lb = options["D"] * np.maximum(options["K"] - options["F"], 0.0)
    put_ub = options["D"] * options["K"]
    lb = np.where(options["cp_flag"].eq("C"), call_lb, put_lb)
    ub = np.where(options["cp_flag"].eq("C"), call_ub, put_ub)
    options = options.loc[
        (options["mid"] >= lb - 1e-6) & (options["mid"] <= ub + 1e-6)
    ].copy()

    return options.sort_values(
        ["date", "exdate", "am_settlement", "K", "cp_flag"]
    ).reset_index(drop=True)


def build_otm_surface(quotes):
    idx = [
        "date", "exdate", "am_settlement", "K", "T", "S", "F",
        "r", "q", "carry", "D", "k"
    ]
    vals = [
        "mid", "bid", "ask", "spread", "relative_spread",
        "iv_market", "delta_market", "optionid"
    ]

    wide = quotes.pivot_table(
        index=idx, columns="cp_flag", values=vals, aggfunc="first"
    )
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    strict_call = wide["K"] >= wide["F"]
    near = wide["k"].abs() <= ATM_TIGHTER_SIDE_BAND
    have_c = wide.get("iv_market_C", pd.Series(np.nan, index=wide.index)).notna()
    have_p = wide.get("iv_market_P", pd.Series(np.nan, index=wide.index)).notna()
    c_spread = wide.get("spread_C", pd.Series(np.inf, index=wide.index))
    p_spread = wide.get("spread_P", pd.Series(np.inf, index=wide.index))

    choose_call = strict_call.copy()
    both_near = near & have_c & have_p
    choose_call = np.where(both_near, c_spread <= p_spread, choose_call)
    wide["cp_otm"] = np.where(choose_call, "C", "P")

    for base in vals:
        c = wide.get(f"{base}_C", pd.Series(np.nan, index=wide.index))
        p = wide.get(f"{base}_P", pd.Series(np.nan, index=wide.index))
        wide[f"{base}_otm"] = np.where(choose_call, c, p)

    wide = wide.dropna(subset=["iv_market_otm", "mid_otm", "optionid_otm"]).copy()

    sizes = wide.groupby(["date", "exdate", "am_settlement"]).size()
    valid = sizes[sizes >= MIN_POINTS_PER_SLICE].index

    return (
        wide.set_index(["date", "exdate", "am_settlement"])
        .loc[valid]
        .reset_index()
        .sort_values(["date", "T", "k"])
        .reset_index(drop=True)
    )


quotes_all = prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw)
otm = build_otm_surface(quotes_all)

print("Complete cleaned quotes:", len(quotes_all))
print("OTM calibration observations:", len(otm))
print("Trading dates:", otm["date"].nunique())
print("Expiry/settlement slices:", otm.groupby(["date", "exdate", "am_settlement"]).ngroups)


options : one_month_spx_option_price.csv
forwards: one_month_spx_forward_price.csv
spot    : one_month_spx_security_price.csv
rates   : one_month_spx_zero_coupon_yield_curve.csv
Complete cleaned quotes: 192031
OTM calibration observations: 91930
Trading dates: 20
Expiry/settlement slices: 809


## 3. Curves, selected surface and deterministic 80/20 hold-out


In [5]:
@dataclass
class DailyMarketCurve:
    date: pd.Timestamp
    am_settlement: int
    spot: float
    T_nodes: np.ndarray
    logD_nodes: np.ndarray
    logF_nodes: np.ndarray

    def __post_init__(self):
        self._logD = PchipInterpolator(self.T_nodes, self.logD_nodes, extrapolate=True)
        self._logF = PchipInterpolator(self.T_nodes, self.logF_nodes, extrapolate=True)

    def log_discount(self, t):
        return self._logD(np.maximum(np.asarray(t, float), 0.0))

    def log_forward_ratio(self, t):
        return self._logF(np.maximum(np.asarray(t, float), 0.0))

    def discount(self, t0, t1):
        return np.exp(self.log_discount(t1) - self.log_discount(t0))

    def forward(self, t):
        return self.spot * np.exp(self.log_forward_ratio(t))

    def q_integral(self, t0, t1):
        return (
            -(self.log_discount(t1) - self.log_discount(t0))
            -(self.log_forward_ratio(t1) - self.log_forward_ratio(t0))
        )


class CurveBook:
    def __init__(self, spot_raw, forwards_raw, rates_raw):
        self.spot = spot_raw.loc[
            spot_raw["secid"] == SECID, ["date", "close"]
        ].copy()
        self.spot["date"] = pd.to_datetime(self.spot["date"])
        self.spot = self.spot.set_index("date")["close"].to_dict()

        self.forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
        self.forwards["date"] = pd.to_datetime(self.forwards["date"])
        self.forwards["expiration"] = pd.to_datetime(self.forwards["expiration"])

        self.rates = rates_raw.copy()
        self.rates["date"] = pd.to_datetime(self.rates["date"])
        self.rates["rate_decimal"] = pd.to_numeric(
            self.rates["rate"], errors="coerce"
        ) / 100.0
        self._cache = {}

    def get(self, date, am_settlement):
        date = pd.Timestamp(date)
        am = int(am_settlement)
        key = (date, am)
        if key in self._cache:
            return self._cache[key]

        S0 = float(self.spot[date])

        rg = self.rates.loc[
            self.rates["date"] == date, ["days", "rate_decimal"]
        ].dropna().sort_values("days")
        T_r = rg["days"].to_numpy(float) / 365.0
        logD = -rg["rate_decimal"].to_numpy(float) * T_r

        fg = self.forwards.loc[
            (self.forwards["date"] == date)
            & (self.forwards["AMSettlement"] == am),
            ["expiration", "ForwardPrice"],
        ].copy()
        if len(fg) < 2:
            fg = self.forwards.loc[
                self.forwards["date"] == date,
                ["expiration", "ForwardPrice"],
            ].copy()

        fg["T"] = (fg["expiration"] - date).dt.days / 365.0
        fg = (
            fg.loc[(fg["T"] >= 0) & (fg["ForwardPrice"] > 0)]
            .groupby("T", as_index=False)["ForwardPrice"]
            .mean()
            .sort_values("T")
        )
        T_f = fg["T"].to_numpy(float)
        logF = np.log(fg["ForwardPrice"].to_numpy(float) / S0)
        positive = T_f > 0
        T_f, logF = T_f[positive], logF[positive]

        T_all = np.unique(np.concatenate([[0.0], T_r, T_f]))
        logD_all = np.interp(T_all, np.r_[0.0, T_r], np.r_[0.0, logD])
        logF_all = np.interp(T_all, np.r_[0.0, T_f], np.r_[0.0, logF])

        curve = DailyMarketCurve(date, am, S0, T_all, logD_all, logF_all)
        self._cache[key] = curve
        return curve


curve_book = CurveBook(spot_raw, forwards_raw, rates_raw)




def normal_pdf(x):
    x = np.asarray(x, float)
    return np.exp(-0.5*x*x) / math.sqrt(2.0*math.pi)


def bs_forward_vega(F, K, D, T, sigma):
    F, K, D, T, sigma = map(lambda x: np.asarray(x, float), (F,K,D,T,sigma))
    rootT = np.sqrt(np.maximum(T, 1e-12))
    volT = np.maximum(sigma, 1e-8) * rootT
    d1 = (np.log(np.maximum(F,1e-12)/np.maximum(K,1e-12)) + 0.5*volT*volT) / volT
    return D * F * normal_pdf(d1) * rootT


def sample_calibration_day(day, max_per_slice=MAX_QUOTES_PER_SLICE):
    pieces = []
    for _, g in day.groupby(["exdate", "am_settlement"]):
        g = g.sort_values("k")
        if len(g) <= max_per_slice:
            pieces.append(g)
        else:
            pos = np.unique(np.round(np.linspace(0, len(g)-1, max_per_slice)).astype(int))
            pieces.append(g.iloc[pos])
    return pd.concat(pieces, ignore_index=True)


def prepare_selected_universe(date):
    day = otm.loc[otm["date"] == pd.Timestamp(date)].copy()
    selected = sample_calibration_day(day).rename(columns={
        "mid_otm": "target_mid",
        "cp_otm": "target_cp",
        "optionid_otm": "optionid",
    }).reset_index(drop=True)
    selected["target_vega"] = bs_forward_vega(
        selected["F"], selected["K"], selected["D"], selected["T"], selected["iv_market_otm"]
    )
    selected["full_selected_id"] = np.arange(len(selected), dtype=int)
    return selected


def stratified_holdout(selected, fraction=HELDOUT_FRACTION, seed=SEED+77):
    """Approximately 80/20 in every expiry/settlement slice, spread across moneyness."""
    selected = selected.copy().reset_index(drop=True)
    is_holdout = np.zeros(len(selected), dtype=bool)
    rng = np.random.default_rng(seed)
    stride = max(2, int(round(1.0/fraction)))

    for _, g in selected.groupby(["exdate", "am_settlement"], sort=False):
        ordered = g.sort_values("k").index.to_numpy()
        offset = int(rng.integers(0, stride))
        chosen = ordered[offset::stride]
        if len(chosen) == 0:
            chosen = ordered[[len(ordered)//2]]
        if len(chosen) >= len(ordered):
            chosen = chosen[:-1]
        is_holdout[chosen] = True

    selected["split"] = np.where(is_holdout, "holdout", "train")
    train = selected.loc[~is_holdout].copy().reset_index(drop=True)
    holdout = selected.loc[is_holdout].copy().reset_index(drop=True)
    train["contract_id"] = np.arange(len(train), dtype=int)
    return selected, train, holdout


selected_all = prepare_selected_universe(EXAMPLE_DATE)
selected_all, selected_train, selected_holdout = stratified_holdout(selected_all)

print("Selected universe:", len(selected_all))
print("Train / holdout:", len(selected_train), "/", len(selected_holdout))
print("Holdout fraction:", round(len(selected_holdout)/len(selected_all), 4))
print("Distinct maturities in train / holdout:", selected_train["T"].nunique(), "/", selected_holdout["T"].nunique())


Selected universe: 997
Train / holdout: 798 / 199
Holdout fraction: 0.1996
Distinct maturities in train / holdout: 36 / 36


## 4. Neural LV architectures and v5 warm-start

The main run keeps the compact v5 shared $\sigma_\theta(t,X)$. The only architecture ablation is a **shared 4×50 trunk plus small maturity-specific output heads**, which recovers some of Sabaté's interval flexibility without creating a full independent network per SPX maturity.


In [6]:
class FeedForwardTrunk(nn.Module):
    def __init__(self, input_dim, hidden_layers, width):
        super().__init__()
        layers, d = [], input_dim
        for _ in range(hidden_layers):
            layers += [nn.Linear(d, width), nn.ReLU()]
            d = width
        self.net = nn.Sequential(*layers)
        self.output_dim = width

    def forward(self, x):
        return self.net(x)


def sabate_weight_initialisation(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_normal_(module.weight.data, gain=1.5)


def _tx(t, x):
    if x.ndim == 1:
        x = x[:, None]
    if not torch.is_tensor(t):
        t = torch.full_like(x, float(t))
    elif t.ndim == 0:
        t = torch.full_like(x, float(t.item()))
    elif t.ndim == 1:
        t = t[:, None]
    return torch.cat([t, x], dim=1)


class SharedNeuralLocalVol(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = FeedForwardTrunk(2, SDE_HIDDEN_LAYERS, SDE_WIDTH)
        self.out = nn.Sequential(nn.Linear(SDE_WIDTH, 1), nn.Softplus())
        self.apply(sabate_weight_initialisation)

    def forward(self, t, x):
        return self.out(self.trunk(_tx(t, x)))


class MaturityHeadNeuralLocalVol(nn.Module):
    def __init__(self, maturity_boundaries):
        super().__init__()
        self.boundaries = np.asarray(maturity_boundaries, float)
        self.trunk = FeedForwardTrunk(2, SDE_HIDDEN_LAYERS, SDE_WIDTH)
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(SDE_WIDTH, 1), nn.Softplus())
            for _ in self.boundaries
        ])
        self.apply(sabate_weight_initialisation)

    def interval_index(self, t):
        tt = float(t.reshape(-1)[0].item()) if torch.is_tensor(t) else float(t)
        idx = int(np.searchsorted(self.boundaries, tt + 1e-7, side="right"))
        return min(idx, len(self.heads)-1)

    def forward(self, t, x):
        h = self.trunk(_tx(t, x))
        return self.heads[self.interval_index(t)](h)


def make_model(arch, maturity_boundaries):
    if arch == "shared":
        return SharedNeuralLocalVol()
    if arch == "maturity_heads":
        return MaturityHeadNeuralLocalVol(maturity_boundaries)
    raise ValueError(f"Unknown MODEL_ARCH={arch!r}")


class PerOptionHedgeNet(nn.Module):
    def __init__(self, n_options):
        super().__init__()
        self.trunk = FeedForwardTrunk(2, HEDGE_HIDDEN_LAYERS, HEDGE_WIDTH)
        self.output = nn.Linear(HEDGE_WIDTH, int(n_options))
        self.apply(sabate_weight_initialisation)

    def forward_ids(self, t, x, option_ids):
        h = self.trunk(_tx(t, x))
        ids = torch.as_tensor(option_ids, dtype=torch.long, device=h.device)
        return h @ self.output.weight[ids].T + self.output.bias[ids][None, :]


class IntervalHedgeBank(nn.Module):
    def __init__(self, maturity_boundaries, n_options):
        super().__init__()
        self.boundaries = np.asarray(maturity_boundaries, float)
        self.nets = nn.ModuleList([PerOptionHedgeNet(n_options) for _ in self.boundaries])

    def interval_index(self, t):
        tt = float(t.reshape(-1)[0].item()) if torch.is_tensor(t) else float(t)
        idx = int(np.searchsorted(self.boundaries, tt + 1e-7, side="right"))
        return min(idx, len(self.nets)-1)

    def forward_ids(self, t, x, option_ids):
        return self.nets[self.interval_index(t)].forward_ids(t, x, option_ids)


def set_requires_grad(module, value):
    for p in module.parameters():
        p.requires_grad_(value)


def safe_torch_load(path, map_location=DEVICE):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def locate_v5_checkpoint(date):
    stamp = f"{pd.Timestamp(date):%Y%m%d}"
    exact = V5_CACHE_DIR / f"stage1_shared_{stamp}_resume.pt"
    if exact.exists():
        return exact
    matches = sorted(V5_CACHE_DIR.glob(f"*{stamp}*resume*.pt")) if V5_CACHE_DIR.exists() else []
    return matches[0] if matches else None


def warm_start_model_from_v5(model, arch, date):
    path = locate_v5_checkpoint(date)
    if path is None:
        print("No v5 checkpoint found: cold start.")
        return False

    payload = safe_torch_load(path, map_location="cpu")
    state = payload.get("final_model_state") or payload.get("model_state")
    if state is None:
        print("v5 checkpoint has no model state: cold start.")
        return False

    if arch == "shared":
        model.load_state_dict(state, strict=True)
    else:
        trunk_state = {k[len("trunk."):]: v for k,v in state.items() if k.startswith("trunk.")}
        model.trunk.load_state_dict(trunk_state, strict=True)
        w, b = state.get("out.0.weight"), state.get("out.0.bias")
        if w is not None and b is not None:
            with torch.no_grad():
                for head in model.heads:
                    head[0].weight.copy_(w)
                    head[0].bias.copy_(b)

    print("Warm-started model weights from:", path)
    return True


## 5. Calibration context, vega-normalised loss and Sabaté control variates


In [7]:
def build_time_grid(maturities, steps_per_year=BASE_STEPS_PER_YEAR):
    maturities = np.sort(np.unique(np.asarray(maturities, float)))
    max_T = float(maturities[-1])
    base = np.arange(0.0, max_T + 0.5/steps_per_year, 1.0/steps_per_year)
    base = base[base <= max_T + 1e-12]
    grid = np.unique(np.r_[0.0, base, maturities, max_T])
    grid.sort()
    return grid


def nearest_grid_indices(grid, times):
    grid, times = np.asarray(grid), np.asarray(times)
    idx = np.clip(np.searchsorted(grid, times), 0, len(grid)-1)
    left = np.maximum(idx-1, 0)
    idx = np.where(np.abs(grid[left]-times) < np.abs(grid[idx]-times), left, idx)
    if np.max(np.abs(grid[idx]-times)) > 1e-10:
        raise RuntimeError("A listed maturity was not represented exactly on the simulation grid.")
    return idx


@dataclass
class CalibrationContext:
    date: pd.Timestamp
    spot: float
    contracts: pd.DataFrame
    maturity_boundaries: np.ndarray
    grid_np: np.ndarray
    grid: torch.Tensor
    maturity_index: torch.Tensor
    stage_index: np.ndarray
    stage_tensor: torch.Tensor
    K: torch.Tensor
    F_T: torch.Tensor
    D_T: torch.Tensor
    target: torch.Tensor
    vega_scale: torch.Tensor
    cp_sign: torch.Tensor
    am: torch.Tensor
    F_grid: torch.Tensor
    D_grid: torch.Tensor
    loss_mode: str


def build_calibration_context(date, contracts, loss_mode="vega"):
    date = pd.Timestamp(date)
    contracts = contracts.copy().reset_index(drop=True)
    contracts["contract_id"] = np.arange(len(contracts), dtype=int)
    boundaries = np.sort(contracts["T"].unique())
    grid_np = build_time_grid(boundaries)
    maturity_idx = nearest_grid_indices(grid_np, contracts["T"].to_numpy(float))
    stage_map = {round(float(T),12): i for i,T in enumerate(boundaries)}
    stage_index = np.array([stage_map[round(float(T),12)] for T in contracts["T"]], dtype=int)

    if loss_mode == "vega":
        positive = contracts.loc[contracts["target_vega"] > 0, "target_vega"].to_numpy(float)
        floor = float(np.quantile(positive, VEGA_FLOOR_QUANTILE))
        contracts["vega_scale"] = np.maximum(contracts["target_vega"].to_numpy(float), floor)
        print(f"Vega floor ({VEGA_FLOOR_QUANTILE:.0%} quantile): {floor:.4f}")
    else:
        contracts["vega_scale"] = 1.0

    S0 = float(contracts["S"].iloc[0])
    F_grid = np.empty((2, len(grid_np)))
    D_grid = np.empty((2, len(grid_np)))
    for am in [0,1]:
        curve = curve_book.get(date, am)
        F_grid[am] = curve.forward(grid_np)
        D_grid[am] = curve.discount(0.0, grid_np)

    return CalibrationContext(
        date=date, spot=S0, contracts=contracts, maturity_boundaries=boundaries,
        grid_np=grid_np, grid=torch.tensor(grid_np, dtype=torch.float32, device=DEVICE),
        maturity_index=torch.tensor(maturity_idx, dtype=torch.long, device=DEVICE),
        stage_index=stage_index,
        stage_tensor=torch.tensor(stage_index, dtype=torch.long, device=DEVICE),
        K=torch.tensor(contracts["K"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        F_T=torch.tensor(contracts["F"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        D_T=torch.tensor(contracts["D"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        target=torch.tensor(contracts["target_mid"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        vega_scale=torch.tensor(contracts["vega_scale"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        cp_sign=torch.tensor(np.where(contracts["target_cp"].eq("C"),1.0,-1.0), dtype=torch.float32, device=DEVICE),
        am=torch.tensor(contracts["am_settlement"].to_numpy(int), dtype=torch.long, device=DEVICE),
        F_grid=torch.tensor(F_grid, dtype=torch.float32, device=DEVICE),
        D_grid=torch.tensor(D_grid, dtype=torch.float32, device=DEVICE),
        loss_mode=loss_mode,
    )


def antithetic_normals(total_paths, n_steps, seed):
    half = (total_paths+1)//2
    gen = torch.Generator(device="cpu"); gen.manual_seed(int(seed))
    z = torch.randn(half, n_steps, generator=gen, dtype=torch.float32)
    return torch.cat([z,-z], dim=0)[:total_paths].to(DEVICE)


def empirical_payoff_mean(samples, strike_norm, cp_flag):
    x = np.sort(np.asarray(samples,float)); n = len(x)
    prefix = np.r_[0.0, np.cumsum(x)]; total = prefix[-1]
    k = np.asarray(strike_norm,float); idx = np.searchsorted(x,k,side="right")
    below, above = prefix[idx], total-prefix[idx]
    call = (above-k*(n-idx))/n
    put = (k*idx-below)/n
    return np.where(np.asarray(cp_flag)=="C", call, put)


def maturity_balanced_mean(values, stages):
    return torch.stack([values[stages == s].mean() for s in torch.unique(stages)]).mean()


def simulate_estimators(model, hedge, ctx, contract_ids, z, phase):
    ids_np = np.asarray(contract_ids,dtype=int)
    ids = torch.as_tensor(ids_np,dtype=torch.long,device=DEVICE)
    maturity_idx = ctx.maturity_index[ids]
    K, F_T, D_T = ctx.K[ids], ctx.F_T[ids], ctx.D_T[ids]
    cp, am = ctx.cp_sign[ids], ctx.am[ids]

    B, M = z.shape[0], len(ids_np)
    x = torch.ones((B,1),dtype=torch.float32,device=DEVICE)
    cv = torch.zeros((B,M),dtype=torch.float32,device=DEVICE)
    raw = torch.zeros((B,M),dtype=torch.float32,device=DEVICE)

    for step in range(int(maturity_idx.max().item())):
        t = ctx.grid[step]; dt = ctx.grid[step+1]-ctx.grid[step]; sqrt_dt = torch.sqrt(dt)
        sigma = model(t,x) if phase == "sde" else model(t,x).detach()
        dW = sqrt_dt*z[:,step:step+1]

        active = torch.where(maturity_idx > step)[0]
        if len(active):
            active_ids = ids_np[active.detach().cpu().numpy()]
            h = hedge.forward_ids(t,x.detach(),active_ids)
            if phase == "sde":
                h = h.detach()
            stochastic_piece = x.detach()*sigma.detach()*dW
            discounted_forward = ctx.D_grid[am[active],step]*ctx.F_grid[am[active],step]
            cv[:,active] += h*stochastic_piece*discounted_forward[None,:]

        a = x*sigma
        x = x + a*dW/(1.0 + torch.abs(x.detach()*sigma.detach())*sqrt_dt)

        matured = torch.where(maturity_idx == step+1)[0]
        if len(matured):
            S_T = x*F_T[matured][None,:]; strike = K[matured][None,:]
            call = torch.clamp(S_T-strike,min=0.0); put = torch.clamp(strike-S_T,min=0.0)
            raw[:,matured] = torch.where((cp[matured]>0)[None,:],call,put)*D_T[matured][None,:]

    return raw, raw-cv


class MaturityBalancedSampler:
    def __init__(self, ctx, per_maturity, seed):
        self.groups = [np.where(ctx.stage_index==j)[0] for j in range(len(ctx.maturity_boundaries))]
        self.per_maturity = per_maturity
        self.rng = np.random.default_rng(seed)

    def draw(self):
        out = []
        for g in self.groups:
            n = min(self.per_maturity, len(g))
            out.extend(self.rng.choice(g,size=n,replace=False))
        return np.asarray(out,dtype=int)


@torch.no_grad()
def simulate_terminal_x(model, ctx, z):
    x = torch.ones((z.shape[0],1),dtype=torch.float32,device=DEVICE)
    stage_by_grid = {int(nearest_grid_indices(ctx.grid_np,[T])[0]): j for j,T in enumerate(ctx.maturity_boundaries)}
    terminal = {}
    for step in range(len(ctx.grid_np)-1):
        t = ctx.grid[step]; dt = ctx.grid[step+1]-ctx.grid[step]; sqrt_dt = torch.sqrt(dt)
        sigma = model(t,x); dW = sqrt_dt*z[:,step:step+1]
        a = x*sigma; x = x + a*dW/(1.0+torch.abs(a)*sqrt_dt)
        if step+1 in stage_by_grid:
            terminal[stage_by_grid[step+1]] = x.squeeze(1).detach().cpu().numpy().copy()
    return terminal


def selected_prices_from_terminal(ctx, terminal):
    rows = []
    for stage in range(len(ctx.maturity_boundaries)):
        ids = np.where(ctx.stage_index==stage)[0]
        g = ctx.contracts.iloc[ids]
        payoff = empirical_payoff_mean(
            terminal[stage], g["K"].to_numpy(float)/g["F"].to_numpy(float), g["target_cp"].to_numpy()
        )
        price = g["D"].to_numpy(float)*g["F"].to_numpy(float)*payoff
        tmp = g[["contract_id","optionid","T","k","target_cp","target_mid","vega_scale"]].copy()
        tmp["raw_model_price"] = price
        tmp["raw_error"] = price-tmp["target_mid"].to_numpy(float)
        rows.append(tmp)
    return pd.concat(rows,ignore_index=True)


def objective_diagnostics(model, ctx, z_fixed):
    table = selected_prices_from_terminal(ctx, simulate_terminal_x(model,ctx,z_fixed))
    e = table["raw_error"].to_numpy(float)
    if ctx.loss_mode == "vega":
        table["objective_error"] = e/table["vega_scale"].to_numpy(float)
    else:
        table["objective_error"] = e/ctx.spot
    by_T = table.groupby("T")["objective_error"].apply(lambda x: float(np.mean(np.asarray(x,float)**2)))
    return {
        "objective_rmse": float(np.sqrt(by_T.mean())),
        "raw_rmse": float(np.sqrt(np.mean(e**2))),
        "raw_mae": float(np.mean(np.abs(e))),
        "table": table,
    }


def choose_even_ids(n_total, n):
    ids = np.arange(n_total,dtype=int)
    if n is None or n >= n_total:
        return ids
    pos = np.unique(np.round(np.linspace(0,n_total-1,n)).astype(int))
    return ids[pos]


@torch.no_grad()
def cv_diagnostics(model, hedge, ctx, ids, z):
    raw, cv = simulate_estimators(model,hedge,ctx,ids,z,phase="sde")
    raw_var, cv_var = raw.var(0,unbiased=False), cv.var(0,unbiased=False)
    return float(torch.median(cv_var/torch.clamp(raw_var,min=1e-12)).cpu())


calib_ctx = build_calibration_context(EXAMPLE_DATE, selected_train, loss_mode="vega")
print("Training contracts:", len(calib_ctx.contracts))
print("Simulation steps:", len(calib_ctx.grid_np)-1)
print("Maturity intervals:", len(calib_ctx.maturity_boundaries))


Vega floor (10% quantile): 227.1186
Training contracts: 798
Simulation steps: 129
Maturity intervals: 36


## 6. Main calibration — vega-normalised loss, resumable, v5-aware, independent checkpoint selection

The hold-out set is never used for optimisation or checkpoint selection. A completed v6 checkpoint is reused first; otherwise the shared v5 model weights are used as an initialisation when available. The hedge/control-variate networks remain fresh because the 80/20 contract set changed.


In [8]:
@dataclass(frozen=True)
class TrainConfig:
    epochs: int
    paths_per_step: int
    steps_per_epoch: int
    quotes_per_maturity: int
    monitor_paths: int
    eval_every_sde: int
    top_k: int
    rerank_paths: int
    save_every: int
    lr_milestones: tuple
    seed_offset: int = 0


def checkpoint_file(run_tag, arch):
    return CACHE_DIR / f"{run_tag}_{arch}_resume.pt"


def candidate_folder(run_tag, arch):
    folder = CACHE_DIR / f"candidates_{run_tag}_{arch}"
    folder.mkdir(exist_ok=True)
    return folder


def save_resume_checkpoint(path, next_epoch, model, hedge, opt_sde, opt_hedge, scheduler,
                           history, candidates, arch, training_finished=False, completed=False,
                           final_model_state=None, final_hedge_state=None, rerank_records=None):
    torch.save({
        "version":"v6", "arch":arch, "training_finished":bool(training_finished),
        "completed":bool(completed), "next_epoch":int(next_epoch),
        "model_state":model.state_dict(), "hedge_state":hedge.state_dict(),
        "opt_sde":opt_sde.state_dict(), "opt_hedge":opt_hedge.state_dict(),
        "scheduler":scheduler.state_dict(), "history":history, "candidates":candidates,
        "final_model_state":final_model_state, "final_hedge_state":final_hedge_state,
        "rerank_records":rerank_records,
    }, path)


def maybe_store_top_candidate(model, hedge, candidates, epoch, diag, run_tag, arch, top_k):
    score = float(diag["objective_rmse"])
    if len(candidates) >= top_k and score >= max(c["monitor_objective_rmse"] for c in candidates):
        return candidates
    folder = candidate_folder(run_tag, arch)
    cpath = folder / f"candidate_epoch_{int(epoch):04d}.pt"
    torch.save({
        "epoch":int(epoch), "monitor_objective_rmse":score,
        "monitor_raw_rmse":float(diag["raw_rmse"]),
        "model_state":model.state_dict(), "hedge_state":hedge.state_dict(),
    }, cpath)
    candidates = list(candidates) + [{
        "epoch":int(epoch), "monitor_objective_rmse":score,
        "monitor_raw_rmse":float(diag["raw_rmse"]), "path":str(cpath),
    }]
    candidates = sorted(candidates, key=lambda x: x["monitor_objective_rmse"])
    while len(candidates) > top_k:
        removed = candidates.pop(-1)
        try:
            Path(removed["path"]).unlink(missing_ok=True)
        except OSError:
            pass
    return candidates


@torch.no_grad()
def rerank_candidates(ctx, candidates, arch, cfg):
    if not candidates:
        raise RuntimeError("No monitor-selected checkpoints available to rerank.")
    z = antithetic_normals(cfg.rerank_paths, len(ctx.grid_np)-1, SEED+5_000+cfg.seed_offset)
    temp = make_model(arch, ctx.maturity_boundaries).to(DEVICE).eval()
    rows = []
    for c in tqdm(candidates, desc=f"Rerank {arch}", unit="model"):
        p = safe_torch_load(c["path"])
        temp.load_state_dict(p["model_state"])
        d = objective_diagnostics(temp, ctx, z)
        rows.append({**c, "rerank_objective_rmse":d["objective_rmse"], "rerank_raw_rmse":d["raw_rmse"]})
    del z, temp; gc.collect()
    table = pd.DataFrame(rows).sort_values(["rerank_objective_rmse","monitor_objective_rmse"]).reset_index(drop=True)
    table["independent_rank"] = np.arange(1,len(table)+1)
    return table, table.iloc[0].to_dict()


def train_v6(ctx, arch, cfg, run_tag, warm_start_v5=False):
    path = checkpoint_file(run_tag, arch)
    model = make_model(arch, ctx.maturity_boundaries).to(DEVICE)
    hedge = IntervalHedgeBank(ctx.maturity_boundaries, len(ctx.contracts)).to(DEVICE)
    opt_sde = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    opt_hedge = torch.optim.Adam(hedge.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(opt_sde, milestones=list(cfg.lr_milestones), gamma=0.2)

    start_epoch, history, candidates, training_finished = 0, [], [], False
    if path.exists() and not FORCE_RETRAIN:
        p = safe_torch_load(path)
        if p.get("completed",False):
            model.load_state_dict(p["final_model_state"]); hedge.load_state_dict(p["final_hedge_state"])
            model.eval(); hedge.eval(); set_requires_grad(model,False); set_requires_grad(hedge,False)
            print("Loaded completed v6 checkpoint:", path)
            return model, hedge, pd.DataFrame(p["history"]), pd.DataFrame(p["rerank_records"])
        model.load_state_dict(p["model_state"]); hedge.load_state_dict(p["hedge_state"])
        opt_sde.load_state_dict(p["opt_sde"]); opt_hedge.load_state_dict(p["opt_hedge"])
        scheduler.load_state_dict(p["scheduler"])
        start_epoch = int(p["next_epoch"]); history = list(p["history"])
        candidates = list(p.get("candidates",[])); training_finished = bool(p.get("training_finished",False))
        print("Resuming v6 at epoch", start_epoch+1 if not training_finished else "reranking")
    elif warm_start_v5:
        warm_start_model_from_v5(model, arch, ctx.date)

    sampler = MaturityBalancedSampler(ctx, cfg.quotes_per_maturity, SEED+10+cfg.seed_offset)
    z_monitor = antithetic_normals(cfg.monitor_paths, len(ctx.grid_np)-1, SEED+100+cfg.seed_offset)

    if not training_finished:
        sde_count = sum(1 for e in range(start_epoch) if e % 2 == 0)
        bar = tqdm(
            range(start_epoch, cfg.epochs),
            desc=f"v6 {run_tag} [{arch}]",
            unit="epoch",
            dynamic_ncols=True,
            mininterval=1.0,
        )
        for epoch in bar:
            phase = "sde" if epoch % 2 == 0 else "hedge"
            set_requires_grad(model, phase=="sde"); set_requires_grad(hedge, phase=="hedge")
            losses = []
            for _ in range(cfg.steps_per_epoch):
                ids = sampler.draw()
                z = torch.randn(cfg.paths_per_step, len(ctx.grid_np)-1, dtype=torch.float32, device=DEVICE)
                opt_sde.zero_grad(set_to_none=True); opt_hedge.zero_grad(set_to_none=True)
                raw, cv = simulate_estimators(model, hedge, ctx, ids, z, phase)
                ids_t = torch.as_tensor(ids,dtype=torch.long,device=DEVICE)
                stages = ctx.stage_tensor[ids_t]

                if phase == "sde":
                    residual = cv.mean(0)-ctx.target[ids_t]
                    if ctx.loss_mode == "vega":
                        residual = residual/ctx.vega_scale[ids_t]
                    else:
                        residual = residual/ctx.spot
                    loss = maturity_balanced_mean(residual.square(), stages)
                    loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),SDE_GRAD_CLIP); opt_sde.step()
                else:
                    var = torch.var(cv/ctx.spot,dim=0,unbiased=False)
                    loss = maturity_balanced_mean(var, stages)
                    loss.backward(); nn.utils.clip_grad_norm_(hedge.parameters(),HEDGE_GRAD_CLIP); opt_hedge.step()

                losses.append(float(loss.detach().cpu())); del raw,cv,z

            scheduler.step(); diag = None
            if phase == "sde":
                sde_count += 1
                if sde_count == 1 or sde_count % cfg.eval_every_sde == 0:
                    set_requires_grad(model,False); set_requires_grad(hedge,False)
                    diag = objective_diagnostics(model,ctx,z_monitor)
                    candidates = maybe_store_top_candidate(model,hedge,candidates,epoch+1,diag,run_tag,arch,cfg.top_k)

            row = {
                "epoch":epoch+1,"phase":phase,"train_loss":float(np.mean(losses)),
                "sde_lr":float(opt_sde.param_groups[0]["lr"]),
                "objective_rmse":np.nan if diag is None else diag["objective_rmse"],
                "raw_rmse":np.nan if diag is None else diag["raw_rmse"],
            }
            history.append(row)
            post = {"phase":phase,"loss":f"{row['train_loss']:.2e}","lr":f"{row['sde_lr']:.1e}"}
            if diag is not None:
                post["obj_RMSE"] = f"{diag['objective_rmse']:.4f}"; post["$RMSE"] = f"{diag['raw_rmse']:.3f}"
            bar.set_postfix(post)

            if (epoch+1)%cfg.save_every==0 or epoch==cfg.epochs-1:
                save_resume_checkpoint(path,epoch+1,model,hedge,opt_sde,opt_hedge,scheduler,history,candidates,arch,
                                    training_finished=(epoch==cfg.epochs-1),completed=False)
            gc.collect()

        training_finished = True
        save_resume_checkpoint(path,cfg.epochs,model,hedge,opt_sde,opt_hedge,scheduler,history,candidates,arch,
                               training_finished=True,completed=False)

    del z_monitor; gc.collect()
    rerank_table,best = rerank_candidates(ctx,candidates,arch,cfg)
    print(f"Independent rerank selected epoch {int(best['epoch'])}: objective={best['rerank_objective_rmse']:.5f}, $RMSE={best['rerank_raw_rmse']:.4f}")
    p = safe_torch_load(best["path"]); model.load_state_dict(p["model_state"]); hedge.load_state_dict(p["hedge_state"])
    model.eval(); hedge.eval(); set_requires_grad(model,False); set_requires_grad(hedge,False)

    final_model_state = copy.deepcopy(model.state_dict()); final_hedge_state = copy.deepcopy(hedge.state_dict())
    save_resume_checkpoint(path,cfg.epochs,model,hedge,opt_sde,opt_hedge,scheduler,history,candidates,arch,
                           training_finished=True,completed=True,final_model_state=final_model_state,
                           final_hedge_state=final_hedge_state,rerank_records=rerank_table.to_dict("records"))
    return model, hedge, pd.DataFrame(history), rerank_table


calibration_date = pd.Timestamp(EXAMPLE_DATE)
dates = sorted(pd.to_datetime(quotes_all["date"].unique()))
next_date = dates[dates.index(calibration_date)+1]
v5_available = locate_v5_checkpoint(calibration_date) is not None
main_epochs = WARM_START_EPOCHS if (WARM_START_FROM_V5 and v5_available) else COLD_START_EPOCHS
MAIN_TRAIN_CFG = TrainConfig(
    epochs=main_epochs, paths_per_step=PATHS_PER_STEP, steps_per_epoch=STEPS_PER_EPOCH,
    quotes_per_maturity=QUOTES_PER_MATURITY_PER_STEP, monitor_paths=FIXED_RAW_EVAL_PATHS,
    eval_every_sde=EVAL_EVERY_SDE_EPOCHS, top_k=TOP_K_CHECKPOINTS, rerank_paths=RERANK_PATHS,
    save_every=RESUME_SAVE_EVERY_EPOCHS,
    lr_milestones=(int(0.5*main_epochs),int(0.8*main_epochs)), seed_offset=0,
)
MAIN_RUN_TAG = f"real_{calibration_date:%Y%m%d}_split80_20"

print("Calibration / next date:", calibration_date.date(), "/", next_date.date())
print("v5 warm-start available:", v5_available, "| v6 epochs:", main_epochs)
model, hedge, training_history, rerank_table = train_v6(
    calib_ctx, MODEL_ARCH, MAIN_TRAIN_CFG, MAIN_RUN_TAG,
    warm_start_v5=(WARM_START_FROM_V5 and v5_available),
)


Calibration / next date: 2024-03-01 / 2024-03-04
v5 warm-start available: True | v6 epochs: 600
Loaded completed v6 checkpoint: neural_lv_v6_cache\real_20240301_split80_20_shared_resume.pt


v6 real_20240301_split80_20 [shared]:  10%|▉         | 56/580 [1:05:56<10:43:59, 73.74s/epoch, phase=hedge, loss=1.33e-04, lr=1.0e-03]
                    
v6 real_20240301_split80_20 [shared]:  10%|▉         | 57/580 [1:06:43<9:33:11, 65.76s/epoch, phase=sde, loss=3.16e-05, lr=1.0e-03] 


v6 real_20240301_split80_20 [shared]:  19%|█▉        | 111/580 [2:07:25<8:25:05, 64.62s/epoch, phase=sde, loss=2.21e-05, lr=1.0e-03] 

v6 real_20240301_split80_20 [shared]:  19%|█▉        | 112/580 [2:08:40<8:47:15, 67.60s/epoch, phase=hedge, loss=7.40e-05, lr=1.0e-03]                            

v6 real_20240301_split80_20 [shared]:  19%|█▉        | 113/580 [2:09:33<8:13:41, 63.43s/epoch, phase=sde, loss=2.04e-05, lr=1.0e-03]                              

v6 real_20240301_split80_20 [shared]:  20%|█▉        | 114/580 [2:10:50<8:44:35, 67.54s/epoch, phase=hedge, loss=7.55e-05, lr=1.0e-03]      


v6 real_20240301_split80_20 [shared]:  51%|█████     | 295/580 [5:38:48<3:28:06, 43.81s/epoch, phase=sde, loss=7.82e-06, lr=2.0e-04]  
v6 real_20240301_split80_20 [shared]:  51%|█████     | 296/580 [5:39:57<4:03:53, 51.53s/epoch, phase=hedge, loss=3.01e-05, lr=2.0e-04]   


v6 real_20240301_split80_20 [shared]: 100%|██████████| 580/580 [8:55:30<00:00, 55.40s/epoch, phase=hedge, loss=2.14e-05, lr=4.0e-05]
Rerank shared: 100%|██████████| 5/5 [00:22<00:00,  4.52s/model]
Independent rerank selected epoch 559: objective=0.00268, $RMSE=1.8802

## 7. Same-day held-out pricing and corrected spot-delta audit

Delta is computed with **contractual strike fixed**. Under a spot bump, deterministic carry is held fixed so the forward scales with spot; the same terminal $X_T$ paths are reused for up/down valuations. This avoids the v5 normalised-strike ambiguity and is much cheaper than simulating three state processes.


In [9]:
def tkey(t):
    return round(float(t),12)


def bs_forward_price(F,K,D,T,sigma,cp="C"):
    F,K,D,T,sigma = map(float,(F,K,D,T,sigma))
    if T <= 0 or sigma <= 0:
        return D*max(F-K,0.0) if cp=="C" else D*max(K-F,0.0)
    srt = sigma*math.sqrt(T)
    d1 = (math.log(F/K)+0.5*srt*srt)/srt; d2 = d1-srt
    call = D*(F*float(ndtr(d1))-K*float(ndtr(d2)))
    return call if cp=="C" else call-D*(F-K)


def implied_vol_from_price(price,F,K,D,T,cp):
    intrinsic = D*max(F-K,0.0) if cp=="C" else D*max(K-F,0.0)
    upper = D*F if cp=="C" else D*K
    if not (intrinsic-1e-8 <= price <= upper+1e-8) or T <= 0:
        return np.nan
    f = lambda s: bs_forward_price(F,K,D,T,s,cp)-price
    try:
        return float(brentq(f,1e-5,5.0,maxiter=100))
    except ValueError:
        return np.nan


@torch.no_grad()
def simulate_same_day_x(model,maturities,total_paths,chunk_paths,seed,progress=True):
    if total_paths%2:
        raise ValueError("total_paths must be even for antithetic evaluation.")
    grid = build_time_grid(maturities)
    maturity_idx = {tkey(T):int(nearest_grid_indices(grid,[T])[0]) for T in np.unique(np.asarray(maturities,float))}
    index_to_keys = {}
    for key,idx in maturity_idx.items(): index_to_keys.setdefault(idx,[]).append(key)
    collected = {key:[] for key in maturity_idx}
    gen = torch.Generator(device="cpu"); gen.manual_seed(seed)
    total_pairs,chunk_pairs,done = total_paths//2,max(1,chunk_paths//2),0
    bar = tqdm(total=total_paths,desc="Final same-day MC",unit="path",disable=not progress)
    while done < total_pairs:
        n_pairs = min(chunk_pairs,total_pairs-done)
        z_half = torch.randn(n_pairs,len(grid)-1,generator=gen,dtype=torch.float32)
        z = torch.cat([z_half,-z_half],0).to(DEVICE); n=2*n_pairs
        x = torch.ones((n,1),dtype=torch.float32,device=DEVICE)
        for step in range(len(grid)-1):
            t=float(grid[step]); dt=float(grid[step+1]-grid[step]); sqrt_dt=math.sqrt(dt)
            sigma=model(torch.full_like(x,t),x); dW=sqrt_dt*z[:,step:step+1]
            a=x*sigma; x=x+a*dW/(1.0+torch.abs(a)*sqrt_dt)
            if step+1 in index_to_keys:
                arr=x.squeeze(1).detach().cpu().numpy()
                paired=np.stack([arr[:n_pairs],arr[n_pairs:]],axis=1)
                for key in index_to_keys[step+1]: collected[key].append(paired.copy())
        done += n_pairs; bar.update(2*n_pairs); del z,x; gc.collect()
    bar.close()
    return {k:np.concatenate(v,axis=0) for k,v in collected.items()}


def price_rows_same_day(rows,samples_by_T,base_col="mid"):
    out=rows.copy().reset_index(drop=True); price=np.empty(len(out))
    for T,idx in out.groupby("T").groups.items():
        pos=np.asarray(list(idx),dtype=int); g=out.loc[idx]; samples=samples_by_T[tkey(T)].reshape(-1)
        payoff=empirical_payoff_mean(samples,g["K"].to_numpy(float)/g["F"].to_numpy(float),g["cp_flag"].to_numpy())
        price[pos]=g["D"].to_numpy(float)*g["F"].to_numpy(float)*payoff
    out["neural_lv_price"]=price; out["pricing_error"]=price-out[base_col].to_numpy(float)
    out["inside_spread"]=out["neural_lv_price"].between(out["bid"],out["ask"])
    return out


def add_model_iv(rows,desc="Model IV"):
    out=rows.copy(); iv=[]
    for r in tqdm(out.itertuples(index=False),total=len(out),desc=desc,unit="opt"):
        iv.append(implied_vol_from_price(r.neural_lv_price,r.F,r.K,r.D,r.T,r.cp_flag))
    out["iv_model"]=iv
    market_col = "iv_market" if "iv_market" in out.columns else "iv_market_otm"
    out["iv_error"] = out["iv_model"]-out[market_col].to_numpy(float)
    return out


def delta_rows_from_terminal(rows,samples_by_T,bump_bps=DELTA_BUMP_BPS):
    out=rows.copy().reset_index(drop=True)
    for bps in bump_bps:
        eps=float(bps)*1e-4; delta=np.empty(len(out))
        for T,idx in out.groupby("T").groups.items():
            pos=np.asarray(list(idx),dtype=int); g=out.loc[idx]; samples=samples_by_T[tkey(T)].reshape(-1)
            F=g["F"].to_numpy(float); K=g["K"].to_numpy(float); D=g["D"].to_numpy(float)
            cp=g["cp_flag"].to_numpy(); S=g["S"].to_numpy(float)
            Fu,Fd=F*(1+eps),F*(1-eps)
            pu=D*Fu*empirical_payoff_mean(samples,K/Fu,cp)
            pdn=D*Fd*empirical_payoff_mean(samples,K/Fd,cp)
            delta[pos]=(pu-pdn)/(2*eps*S)
        out[f"delta_{bps}bp"]=delta
    headline=f"delta_{DELTA_HEADLINE_BPS}bp"; out["neural_lv_delta"]=out[headline]
    others=[f"delta_{b}bp" for b in bump_bps if b!=DELTA_HEADLINE_BPS]
    out["delta_bump_instability"]=np.max(np.column_stack([np.abs(out[c]-out[headline]) for c in others]),axis=1)
    return out


def price_summary(df):
    e=df["pricing_error"].to_numpy(float)
    out={"n":len(df),"RMSE":float(np.sqrt(np.mean(e**2))),"MAE":float(np.mean(np.abs(e))),
         "inside_bid_ask":float(df["inside_spread"].mean())}
    if "iv_error" in df:
        ie=df["iv_error"].dropna().to_numpy(float)
        out["IV_RMSE"]=float(np.sqrt(np.mean(ie**2))) if len(ie) else np.nan
    return out


def bs_delta_unit_test():
    class ConstantVol(nn.Module):
        def __init__(self,sigma): super().__init__(); self.sigma=float(sigma)
        def forward(self,t,x): return torch.full_like(x,self.sigma)
    S,r,q,T,K,sig=5000.0,0.04,0.015,0.5,5100.0,0.20
    F=S*math.exp((r-q)*T); D=math.exp(-r*T)
    samples=simulate_same_day_x(ConstantVol(sig).to(DEVICE),[T],30_000,10_000,SEED+6_600,progress=False)
    row=pd.DataFrame([{"S":S,"F":F,"D":D,"K":K,"T":T,"cp_flag":"C"}])
    mc=float(delta_rows_from_terminal(row,samples)["neural_lv_delta"].iloc[0])
    d1=(math.log(S/K)+(r-q+0.5*sig*sig)*T)/(sig*math.sqrt(T)); exact=math.exp(-q*T)*float(ndtr(d1))
    err=abs(mc-exact); print(f"BS delta unit test: MC={mc:.5f}, analytic={exact:.5f}, abs error={err:.5f}")
    if err>0.03: warnings.warn("Delta unit test is noisier than expected; inspect before trusting hedging results.")
    return err


delta_unit_test_error = bs_delta_unit_test()

day0_full=quotes_all.loc[quotes_all["date"]==calibration_date].copy().reset_index(drop=True)
day0_otm=otm.loc[otm["date"]==calibration_date].rename(columns={
    "mid_otm":"mid","bid_otm":"bid","ask_otm":"ask","cp_otm":"cp_flag","optionid_otm":"optionid"
}).copy().reset_index(drop=True)

samples_same_day=simulate_same_day_x(model,day0_full["T"].unique(),FINAL_MC_PATHS,FINAL_PATH_CHUNK,SEED+7_000)
same_day_full=delta_rows_from_terminal(price_rows_same_day(day0_full,samples_same_day),samples_same_day)
same_day_otm=price_rows_same_day(day0_otm,samples_same_day)

train_ids=set(selected_train["optionid"].astype(int)); holdout_ids=set(selected_holdout["optionid"].astype(int))
same_day_train=add_model_iv(same_day_full.loc[same_day_full["optionid"].astype(int).isin(train_ids)].copy(),"Train IV")
same_day_holdout=add_model_iv(same_day_full.loc[same_day_full["optionid"].astype(int).isin(holdout_ids)].copy(),"Holdout IV")
same_day_otm=add_model_iv(same_day_otm,"Full OTM IV")

hierarchy=pd.DataFrame({
    "selected_train_80":price_summary(same_day_train),
    "selected_holdout_20":price_summary(same_day_holdout),
    "full_OTM_strip":price_summary(same_day_otm),
    "complete_clean_chain":price_summary(same_day_full),
}).T
print("Same-day / held-out hierarchy:")
print(hierarchy.to_string())
print("Median delta bump instability:",float(same_day_full["delta_bump_instability"].median()))

del samples_same_day; gc.collect()


BS delta unit test: MC=0.50156, analytic=0.50382, abs error=0.00226


Full OTM IV: 100%|██████████| 4304/4304 [00:00<00:00, 10035.49opt/s]


Same-day / held-out hierarchy:
                           n      RMSE       MAE  inside_bid_ask   IV_RMSE
selected_train_80      798.0  1.825218  1.312313        0.126566  0.003384
selected_holdout_20    199.0  1.840677  1.274612        0.155779  0.003213
full_OTM_strip        4304.0  1.915141  1.376699        0.121050  0.003167
complete_clean_chain  9046.0  6.155766  4.524003        0.119611       NaN
Median delta bump instability: 3.44371305104306e-05


675

## 8. Frozen next-day pricing with the correct model clock + one-day hedge audit

The learned $\sigma_\theta$ is frozen, but the next day's observed spot/forward/discount curves are used. The neural-network time input starts at elapsed calendar time $\delta$, not at zero. A tiny recording-model unit test guards against accidentally resetting the clock.


In [10]:
def build_evaluation_pairs(quotes):
    trading_dates=sorted(quotes["date"].unique())
    next_map={trading_dates[i]:trading_dates[i+1] for i in range(len(trading_dates)-1)}
    left=quotes.copy(); left["next_date"]=left["date"].map(next_map)
    cols=["date","optionid","cp_flag","K","exdate","am_settlement","mid","bid","ask","S","F","D","r","q","carry","T","k"]
    right=quotes[cols].rename(columns={
        "date":"next_date","mid":"mid_next","bid":"bid_next","ask":"ask_next","S":"S_next",
        "F":"F_next","D":"D_next","r":"r_next","q":"q_next","carry":"carry_next","T":"T_next","k":"k_next",
    })
    return left.merge(right,on=["next_date","optionid","cp_flag","K","exdate","am_settlement"],how="inner",validate="many_to_one")


@torch.no_grad()
def simulate_next_day_y_pairs(model,curve0,curve1,elapsed,maturities_remaining,total_paths,chunk_paths,seed,progress=True):
    if total_paths%2: raise ValueError("total_paths must be even.")
    grid=build_time_grid(maturities_remaining)
    maturity_idx={tkey(T):int(nearest_grid_indices(grid,[T])[0]) for T in np.unique(np.asarray(maturities_remaining,float))}
    index_to_keys={}
    for key,idx in maturity_idx.items(): index_to_keys.setdefault(idx,[]).append(key)
    collected={key:[] for key in maturity_idx}; gen=torch.Generator(device="cpu"); gen.manual_seed(seed)
    total_pairs,chunk_pairs,done=total_paths//2,max(1,chunk_paths//2),0
    bar=tqdm(total=total_paths,desc=f"Next-day MC AM={curve1.am_settlement}",unit="path",disable=not progress)
    while done<total_pairs:
        n_pairs=min(chunk_pairs,total_pairs-done)
        z_half=torch.randn(n_pairs,len(grid)-1,generator=gen,dtype=torch.float32)
        z=torch.cat([z_half,-z_half],0).to(DEVICE); n=2*n_pairs
        y=torch.ones((n,1),dtype=torch.float32,device=DEVICE)  # y=S/F_new(v)
        for step in range(len(grid)-1):
            v=float(grid[step]); dt=float(grid[step+1]-grid[step]); sqrt_dt=math.sqrt(dt)
            abs_t=elapsed+v                              # crucial: frozen model clock does NOT restart
            Fnew=float(curve1.forward(v)); Fold=float(curve0.forward(abs_t))
            x_old=y*(Fnew/Fold)                         # old-model state x=S/F_old(abs_t)
            sigma=model(torch.full_like(y,abs_t),x_old)
            dW=sqrt_dt*z[:,step:step+1]; a=y*sigma
            y=y+a*dW/(1.0+torch.abs(a)*sqrt_dt)
            if step+1 in index_to_keys:
                arr=y.squeeze(1).detach().cpu().numpy(); paired=np.stack([arr[:n_pairs],arr[n_pairs:]],axis=1)
                for key in index_to_keys[step+1]: collected[key].append(paired.copy())
        done+=n_pairs; bar.update(2*n_pairs); del z,y; gc.collect()
    bar.close(); return {k:np.concatenate(v,axis=0) for k,v in collected.items()}


def price_next_day_from_pairs(pairs,samples_by_am):
    out=pairs.copy().reset_index(drop=True); price=np.empty(len(out))
    for (am,T),idx in out.groupby(["am_settlement","T_next"]).groups.items():
        pos=np.asarray(list(idx),dtype=int); g=out.loc[idx]; samples=samples_by_am[int(am)][tkey(T)].reshape(-1)
        payoff=empirical_payoff_mean(samples,g["K"].to_numpy(float)/g["F_next"].to_numpy(float),g["cp_flag"].to_numpy())
        price[pos]=g["D_next"].to_numpy(float)*g["F_next"].to_numpy(float)*payoff
    out["model_price_next"]=price; out["next_day_pricing_error"]=price-out["mid_next"].to_numpy(float)
    out["next_day_inside_bid_ask"]=out["model_price_next"].between(out["bid_next"],out["ask_next"])
    return out


def next_day_clock_unit_test():
    class ToyCurve:
        am_settlement=0
        def __init__(self,spot,carry): self.spot,self.carry=float(spot),float(carry)
        def forward(self,t): return self.spot*np.exp(self.carry*np.asarray(t,float))
    class Recorder(nn.Module):
        def __init__(self): super().__init__(); self.times=[]
        def forward(self,t,x): self.times.append(float(t.reshape(-1)[0])); return torch.full_like(x,0.2)
    elapsed=0.10; rec=Recorder().to(DEVICE)
    simulate_next_day_y_pairs(rec,ToyCurve(100,0.02),ToyCurve(101,0.02),elapsed,[0.08],4,4,SEED+6_700,progress=False)
    err = abs(rec.times[0] - elapsed)
    print(
        f"Next-day clock unit test: first NN time={rec.times[0]:.6f}, "
        f"expected={elapsed:.6f}, error={err:.2e}"
    )
    if err > 1e-6:
        raise RuntimeError("Next-day model clock restarted incorrectly.")
    return err


next_day_clock_error=next_day_clock_unit_test()
evaluation_pairs=build_evaluation_pairs(quotes_all)
transition_pairs=evaluation_pairs.loc[(evaluation_pairs["date"]==calibration_date)&(evaluation_pairs["next_date"]==next_date)].copy().reset_index(drop=True)
print("Identical-contract next-day pairs:",len(transition_pairs))

elapsed=(next_date-calibration_date).days/365.0; samples_next_by_am={}
for am in sorted(transition_pairs["am_settlement"].unique()):
    g=transition_pairs.loc[transition_pairs["am_settlement"]==am]
    samples_next_by_am[int(am)]=simulate_next_day_y_pairs(
        model,curve_book.get(calibration_date,int(am)),curve_book.get(next_date,int(am)),elapsed,g["T_next"].unique(),
        FINAL_MC_PATHS,FINAL_PATH_CHUNK,SEED+8_000+int(am),progress=True)
next_day_results=price_next_day_from_pairs(transition_pairs,samples_next_by_am)

# Fan-style one-day hedge audit using the corrected current-day model delta.
delta_map=same_day_full[["optionid","neural_lv_delta","delta_bump_instability"]].drop_duplicates("optionid")
next_day_results=next_day_results.merge(delta_map,on="optionid",how="left",validate="many_to_one")
next_day_results["dP_market"]=next_day_results["mid_next"]-next_day_results["mid"]
next_day_results["dS"]=next_day_results["S_next"]-next_day_results["S"]
next_day_results["hedge_error"]=next_day_results["dP_market"]-next_day_results["neural_lv_delta"]*next_day_results["dS"]

pe=next_day_results["next_day_pricing_error"].to_numpy(float); he=next_day_results["hedge_error"].dropna().to_numpy(float)
print(f"Next-day pricing: RMSE=${np.sqrt(np.mean(pe**2)):.4f}, MAE=${np.mean(np.abs(pe)):.4f}, inside spread={next_day_results['next_day_inside_bid_ask'].mean():.3f}")
print(f"Corrected one-day hedge: RMSE={np.sqrt(np.mean(he**2)):.4f}, MAE={np.mean(np.abs(he)):.4f}, bias={np.mean(he):.4f}")

del samples_next_by_am; gc.collect()


Next-day clock unit test: first NN time=0.100000, expected=0.100000, error=1.49e-09
Identical-contract next-day pairs: 8771


Next-day MC AM=1: 100%|██████████| 100000/100000 [00:14<00:00, 7030.24path/s]


Next-day pricing: RMSE=$6.6956, MAE=$5.1058, inside spread=0.062
Corrected one-day hedge: RMSE=1.3938, MAE=1.1149, bias=-0.4201


0

## 9. Compact v6 summary and essential exports


In [28]:
FINAL_CV_DIAG_CONTRACTS=64; FINAL_CV_DIAG_PATHS=8_192
cv_ids=choose_even_ids(len(calib_ctx.contracts),FINAL_CV_DIAG_CONTRACTS)
z_cv=antithetic_normals(FINAL_CV_DIAG_PATHS,len(calib_ctx.grid_np)-1,SEED+9_000)
cv_variance_ratio=cv_diagnostics(model,hedge,calib_ctx,cv_ids,z_cv); del z_cv; gc.collect()

best=rerank_table.iloc[0]
next_err=next_day_results["next_day_pricing_error"].to_numpy(float)
hedge_err=next_day_results["hedge_error"].dropna().to_numpy(float)
final_summary={
    "architecture":MODEL_ARCH,
    "v5_warm_start_available":bool(v5_available),
    "training_contracts":len(selected_train),"heldout_contracts":len(selected_holdout),
    "selected_checkpoint_epoch":int(best["epoch"]),
    "independent_rerank_objective_RMSE":float(best["rerank_objective_rmse"]),
    "train_same_day_RMSE":float(hierarchy.loc["selected_train_80","RMSE"]),
    "heldout_same_day_RMSE":float(hierarchy.loc["selected_holdout_20","RMSE"]),
    "heldout_IV_RMSE":float(hierarchy.loc["selected_holdout_20","IV_RMSE"]),
    "heldout_inside_bid_ask":float(hierarchy.loc["selected_holdout_20","inside_bid_ask"]),
    "full_OTM_same_day_RMSE":float(hierarchy.loc["full_OTM_strip","RMSE"]),
    "next_day_RMSE":float(np.sqrt(np.mean(next_err**2))),
    "next_day_MAE":float(np.mean(np.abs(next_err))),
    "one_day_hedge_RMSE":float(np.sqrt(np.mean(hedge_err**2))),
    "one_day_hedge_MAE":float(np.mean(np.abs(hedge_err))),
    "one_day_hedge_bias":float(np.mean(hedge_err)),
    "median_delta_bump_instability":float(same_day_full["delta_bump_instability"].median()),
    "BS_delta_unit_test_abs_error":float(delta_unit_test_error),
    "next_day_clock_unit_test_abs_error":float(next_day_clock_error),
    "cv_median_variance_ratio":float(cv_variance_ratio),
}
print(json.dumps(final_summary,indent=2))

stem=f"{calibration_date:%Y%m%d}_{MODEL_ARCH}_v6"
pd.DataFrame([final_summary]).to_csv(CACHE_DIR/f"summary_{stem}.csv",index=False)
pd.concat([same_day_train.assign(split="train"),same_day_holdout.assign(split="holdout")],ignore_index=True).to_csv(
    CACHE_DIR/f"selected_same_day_{stem}.csv",index=False)
next_day_results.to_csv(CACHE_DIR/f"next_day_pricing_hedge_{stem}.csv",index=False)
print("Saved v6 outputs under:",CACHE_DIR.resolve())


{
  "architecture": "shared",
  "v5_warm_start_available": true,
  "training_contracts": 798,
  "heldout_contracts": 199,
  "selected_checkpoint_epoch": 559,
  "independent_rerank_objective_RMSE": 0.0026832151717308774,
  "train_same_day_RMSE": 1.8252180999907448,
  "heldout_same_day_RMSE": 1.8406774317840258,
  "heldout_IV_RMSE": 0.0032127004664169365,
  "heldout_inside_bid_ask": 0.15577889447236182,
  "full_OTM_same_day_RMSE": 1.915140674595139,
  "next_day_RMSE": 6.6956025589738895,
  "next_day_MAE": 5.105832955771848,
  "one_day_hedge_RMSE": 1.393764459769011,
  "one_day_hedge_MAE": 1.114916438640143,
  "one_day_hedge_bias": -0.4200618758881831,
  "median_delta_bump_instability": 3.44371305104306e-05,
  "BS_delta_unit_test_abs_error": 0.00225702690985774,
  "next_day_clock_unit_test_abs_error": 1.4901161138336505e-09,
  "cv_median_variance_ratio": 0.04053354263305664
}
Saved v6 outputs under: C:\Users\Startklar\ICL\NDEs\Codes\NSDE\neural_lv_v6_cache


## Raw-price loss ablation (seeing if vega-normalized loss is a bottleneck in improving calibration accuracy). No next-day pricing is done yet though.

In [11]:
# ============================================================
# 9A. LOSS ABLATION: raw-price loss, everything else fixed
# ============================================================

RAW_LOSS_ARCH = "shared"
RAW_LOSS_TAG = f"{MAIN_RUN_TAG}_rawloss"

# Same contracts / grid / curves / held-out split as the main v6 run.
# Only the calibration loss changes from Vega-normalised -> raw price.
raw_loss_ctx = build_calibration_context(
    EXAMPLE_DATE,
    selected_train,
    loss_mode="raw",
)

# Reset seeds so this experiment is as comparable as practical.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("RAW-LOSS ABLATION")
print("-----------------")
print("Architecture:", RAW_LOSS_ARCH)
print("Training contracts:", len(raw_loss_ctx.contracts))
print("Held-out contracts:", len(selected_holdout))
print("Epoch budget:", MAIN_TRAIN_CFG.epochs)
print("Warm start from same v5 checkpoint:", WARM_START_FROM_V5 and v5_available)
print("Everything else matches the main v6 training configuration.")

raw_model, raw_hedge, raw_history, raw_rerank = train_v6(
    raw_loss_ctx,
    RAW_LOSS_ARCH,
    MAIN_TRAIN_CFG,
    RAW_LOSS_TAG,
    warm_start_v5=(WARM_START_FROM_V5 and v5_available),
)

RAW-LOSS ABLATION
-----------------
Architecture: shared
Training contracts: 798
Held-out contracts: 199
Epoch budget: 600
Warm start from same v5 checkpoint: True
Everything else matches the main v6 training configuration.
Warm-started model weights from: neural_lv_v5_cache\stage1_shared_20240301_resume.pt


Rerank shared: 100%|██████████| 5/5 [00:18<00:00,  3.66s/model]


Independent rerank selected epoch 499: objective=0.00039, $RMSE=1.9730


v6 real_20240301_split80_20_rawloss [shared]:   0%|          | 3/600 [02:12<7:01:59, 42.41s/epoch, phase=sde, loss=2.51e-06, lr=1.0e-03]
v6 real_20240301_split80_20_rawloss [shared]:   1%|          | 4/600 [03:08<7:57:59, 48.12s/epoch, phase=hedge, loss=1.07e-03, lr=1.0e-03]
v6 real_20240301_split80_20_rawloss [shared]:   2%|▏         | 9/600 [06:02<5:51:02, 35.64s/epoch, phase=sde, loss=5.97e-07, lr=1.0e-03]
v6 real_20240301_split80_20_rawloss [shared]:   2%|▏         | 10/600 [06:48<6:22:10, 38.87s/epoch, phase=hedge, loss=7.06e-04, lr=1.0e-03]
v6 real_20240301_split80_20_rawloss [shared]:   5%|▍         | 29/600 [17:54<5:30:03, 34.68s/epoch, phase=sde, loss=2.61e-07, lr=1.0e-03]
v6 real_20240301_split80_20_rawloss [shared]:   5%|▌         | 30/600 [18:42<6:08:05, 38.75s/epoch, phase=hedge, loss=2.93e-04, lr=1.0e-03]
v6 real_20240301_split80_20_rawloss [shared]:   8%|▊         | 45/600 [28:06<6:31:17, 42.30s/epoch, phase=sde, loss=2.55e-07, lr=1.0e-03]
v6 real_20240301_split80_20_rawloss [shared]:   8%|▊         | 46/600 [29:04<7:14:58, 47.11s/epoch, phase=hedge, loss=2.13e-04, lr=1.0e-03]
v6 real_20240301_split80_20_rawloss [shared]:  10%|█         | 61/600 [37:33<5:12:16, 34.76s/epoch, phase=sde, loss=3.82e-07, lr=1.0e-03]
v6 real_20240301_split80_20_rawloss [shared]:  11%|█         | 64/600 [39:25<5:19:31, 35.77s/epoch, phase=hedge, loss=1.45e-04, lr=1.0e-03]

v6 real_20240301_split80_20_rawloss [shared]: 100%|██████████| 600/600 [5:38:21<00:00, 33.84s/epoch, phase=hedge, loss=2.06e-05, lr=4.0e-05]
Rerank shared: 100%|██████████| 5/5 [00:18<00:00,  3.66s/model]
Independent rerank selected epoch 499: objective=0.00039, $RMSE=1.9730


In [12]:
# ============================================================
# 9A. RAW-LOSS SAME-DAY / HELD-OUT EVALUATION
# ============================================================

# Exact same final Brownian seed and MC budget as the main v6 model.
raw_samples = simulate_same_day_x(
    raw_model,
    day0_full["T"].unique(),
    FINAL_MC_PATHS,
    FINAL_PATH_CHUNK,
    SEED + 7_000,
)

raw_same_day_full = price_rows_same_day(day0_full, raw_samples)
raw_same_day_otm  = price_rows_same_day(day0_otm, raw_samples)

raw_same_day_train = add_model_iv(
    raw_same_day_full.loc[
        raw_same_day_full["optionid"].astype(int).isin(train_ids)
    ].copy(),
    "Raw-loss train IV",
)

raw_same_day_holdout = add_model_iv(
    raw_same_day_full.loc[
        raw_same_day_full["optionid"].astype(int).isin(holdout_ids)
    ].copy(),
    "Raw-loss holdout IV",
)

raw_same_day_otm = add_model_iv(
    raw_same_day_otm,
    "Raw-loss full OTM IV",
)

raw_hierarchy = pd.DataFrame({
    "selected_train_80": price_summary(raw_same_day_train),
    "selected_holdout_20": price_summary(raw_same_day_holdout),
    "full_OTM_strip": price_summary(raw_same_day_otm),
    "complete_clean_chain": price_summary(raw_same_day_full),
}).T

print("\nRAW-PRICE LOSS RESULTS")
print(raw_hierarchy.to_string())

del raw_samples
gc.collect()

Raw-loss full OTM IV: 100%|██████████| 4304/4304 [00:00<00:00, 10872.67opt/s]



RAW-PRICE LOSS RESULTS
                           n      RMSE       MAE  inside_bid_ask   IV_RMSE
selected_train_80      798.0  1.717596  1.229913        0.166667  0.004657
selected_holdout_20    199.0  1.760690  1.205255        0.201005  0.004440
full_OTM_strip        4304.0  1.758901  1.203953        0.172862  0.004085
complete_clean_chain  9046.0  5.973882  4.392464        0.130444       NaN


670

In [13]:
# ============================================================
# 9A. DIRECT LOSS ABLATION COMPARISON
# ============================================================

loss_ablation = pd.DataFrame({
    "v6_vega_loss": {
        "train_RMSE": hierarchy.loc["selected_train_80", "RMSE"],
        "heldout_RMSE": hierarchy.loc["selected_holdout_20", "RMSE"],
        "heldout_MAE": hierarchy.loc["selected_holdout_20", "MAE"],
        "heldout_IV_RMSE": hierarchy.loc["selected_holdout_20", "IV_RMSE"],
        "heldout_inside_bid_ask": hierarchy.loc["selected_holdout_20", "inside_bid_ask"],
        "full_OTM_RMSE": hierarchy.loc["full_OTM_strip", "RMSE"],
    },
    "v6_raw_price_loss": {
        "train_RMSE": raw_hierarchy.loc["selected_train_80", "RMSE"],
        "heldout_RMSE": raw_hierarchy.loc["selected_holdout_20", "RMSE"],
        "heldout_MAE": raw_hierarchy.loc["selected_holdout_20", "MAE"],
        "heldout_IV_RMSE": raw_hierarchy.loc["selected_holdout_20", "IV_RMSE"],
        "heldout_inside_bid_ask": raw_hierarchy.loc["selected_holdout_20", "inside_bid_ask"],
        "full_OTM_RMSE": raw_hierarchy.loc["full_OTM_strip", "RMSE"],
    },
}).T

print("\nVEGA vs RAW-PRICE LOSS — SAME SPLIT, SAME MODEL, SAME MC")
print(loss_ablation.to_string())

loss_ablation.to_csv(
    CACHE_DIR / f"loss_ablation_{calibration_date:%Y%m%d}.csv"
)


VEGA vs RAW-PRICE LOSS — SAME SPLIT, SAME MODEL, SAME MC
                   train_RMSE  heldout_RMSE  heldout_MAE  heldout_IV_RMSE  heldout_inside_bid_ask  full_OTM_RMSE
v6_vega_loss         1.825218      1.840677     1.274612         0.003213                0.155779       1.915141
v6_raw_price_loss    1.717596      1.760690     1.205255         0.004440                0.201005       1.758901


## 10. OPTIONAL — shared vs maturity-head architecture ablation

Run this **after the main result**, not as part of a normal “Run All”. It trains the alternative architecture under the same 80/20 split and CPU budget, then compares both models on the *same* 50k-path held-out evaluation. The alternate model also reuses the v5 shared trunk/output weights where possible.


In [19]:
if not RUN_MATURITY_HEAD_ABLATION:
    print(
        "Architecture ablation skipped. "
        "Set RUN_MATURITY_HEAD_ABLATION=True when ready."
    )
else:
    # IMPORTANT:
    # this is a NEW raw-loss architecture experiment.
    # Do not reuse the earlier Vega-loss maturity-head checkpoint.
    alt_arch = "maturity_heads"
    alt_tag = f"{MAIN_RUN_TAG}_ablation_rawloss"

    alt_model, alt_hedge, alt_history, alt_rerank = train_v6(
        raw_loss_ctx,                # <-- raw loss, same 80/20 contracts
        alt_arch,
        MAIN_TRAIN_CFG,
        alt_tag,                     # <-- NEW tag avoids resuming old Vega run
        warm_start_v5=(WARM_START_FROM_V5 and v5_available),
    )

    def heldout_eval(m, arch, paths=50_000):
        samp = simulate_same_day_x(
            m,
            selected_holdout["T"].unique(),
            paths,
            10_000,
            SEED + 11_000,
            progress=True,
        )

        rows = day0_full.loc[
            day0_full["optionid"].astype(int).isin(holdout_ids)
        ].copy()

        rows = add_model_iv(
            price_rows_same_day(rows, samp),
            f"{arch} heldout IV",
        )

        out = price_summary(rows)
        out["architecture"] = arch

        del samp
        gc.collect()
        return out

    # Compare the RAW-loss shared model against RAW-loss maturity-head model.
    arch_table = pd.DataFrame([
        heldout_eval(raw_model, "shared"),
        heldout_eval(alt_model, "maturity_heads"),
    ]).set_index("architecture")

    print("\nRAW-LOSS ARCHITECTURE ABLATION")
    print(arch_table.to_string())

    # Main criterion = held-out raw-price RMSE.
    winner = arch_table["RMSE"].astype(float).idxmin()
    print("\nLower held-out raw-price RMSE:", winner)

    arch_table.to_csv(
        CACHE_DIR / f"architecture_ablation_rawloss_{calibration_date:%Y%m%d}.csv"
    )

Warm-started model weights from: neural_lv_v5_cache\stage1_shared_20240301_resume.pt


v6 real_20240301_split80_20_ablation_rawloss [maturity_heads]: 100%|██████████| 600/600 [5:39:37<00:00, 33.96s/epoch, phase=hedge, loss=2.11e-05, lr=4.0e-05]                              
Rerank maturity_heads: 100%|██████████| 5/5 [00:20<00:00,  4.19s/model]


Independent rerank selected epoch 479: objective=0.00036, $RMSE=1.8431


maturity_heads heldout IV: 100%|██████████| 199/199 [00:00<00:00, 13031.69opt/s]


RAW-LOSS ARCHITECTURE ABLATION
                  n      RMSE       MAE  inside_bid_ask   IV_RMSE
architecture                                                     
shared          199  1.696436  1.199264        0.155779  0.004184
maturity_heads  199  1.537051  1.044521        0.160804  0.003368

Lower held-out raw-price RMSE: maturity_heads


Warm-started model weights from: neural_lv_v5_cache\stage1_shared_20240301_resume.pt

v6 real_20240301_split80_20_ablation_rawloss [maturity_heads]: 100%|██████████| 600/600 [5:39:37<00:00, 33.96s/epoch, phase=hedge, loss=2.11e-05, lr=4.0e-05]

Rerank maturity_heads: 100%|██████████| 5/5 [00:20<00:00,  4.19s/model]

Independent rerank selected epoch 479: objective=0.00036, $RMSE=1.8431

Final same-day MC: 100%|██████████| 50000/50000 [00:04<00:00, 11190.61path/s]

shared heldout IV: 100%|██████████| 199/199 [00:00<00:00, 7226.74opt/s]

Final same-day MC: 100%|██████████| 50000/50000 [00:04<00:00, 11232.43path/s]

maturity_heads heldout IV: 100%|██████████| 199/199 [00:00<00:00, 13031.69opt/s]

RAW-LOSS ARCHITECTURE ABLATION
                  n      RMSE       MAE  inside_bid_ask   IV_RMSE
architecture                                                     
shared          199  1.696436  1.199264        0.155779  0.004184
maturity_heads  199  1.537051  1.044521        0.160804  0.003368

Lower held-out raw-price RMSE: maturity_heads

## 11. OPTIONAL — reduced Gierjatowicz/Sabaté synthetic Heston replication

This isolates implementation error from real-SPX noise. Targets use Appendix-B Heston parameters $S_0=1, r=0.025, \kappa=0.78, \mu=0.11, \eta=0.68, V_0=0.04, \rho=0.044$, 21 strikes from 0.8 to 1.2 and bimonthly maturities to one year. Target calls are generated semi-analytically and cached. Training remains deliberately CPU-scaled rather than attempting the paper's 40k-path-per-step budget.


In [22]:
if not RUN_SYNTHETIC_REPLICATION:
    print("Synthetic replication skipped. Set RUN_SYNTHETIC_REPLICATION=True and run this cell separately.")
else:
    HESTON=dict(S0=1.0,r=0.025,kappa=0.78,theta=0.11,eta=0.68,v0=0.04,rho=0.044)

    def heston_cf(u,T,p=HESTON):
        iu=1j*u; kappa,theta,eta,v0,rho=p["kappa"],p["theta"],p["eta"],p["v0"],p["rho"]
        d=np.sqrt((rho*eta*iu-kappa)**2+eta**2*(iu+u*u))
        g=(kappa-rho*eta*iu-d)/(kappa-rho*eta*iu+d); ed=np.exp(-d*T)
        C=iu*(math.log(p["S0"])+p["r"]*T)+(kappa*theta/eta**2)*((kappa-rho*eta*iu-d)*T-2*np.log((1-g*ed)/(1-g)))
        D=((kappa-rho*eta*iu-d)/eta**2)*((1-ed)/(1-g*ed))
        return np.exp(C+D*v0)

    def heston_call(K,T,p=HESTON):
        cf_mi=heston_cf(-1j,T,p); logK=math.log(K)
        f1=lambda u: np.real(np.exp(-1j*u*logK)*heston_cf(u-1j,T,p)/(1j*u*cf_mi))
        f2=lambda u: np.real(np.exp(-1j*u*logK)*heston_cf(u,T,p)/(1j*u))
        P1=0.5+quad(f1,1e-8,150,limit=300,epsabs=1e-9,epsrel=1e-7)[0]/math.pi
        P2=0.5+quad(f2,1e-8,150,limit=300,epsabs=1e-9,epsrel=1e-7)[0]/math.pi
        return p["S0"]*P1-K*math.exp(-p["r"]*T)*P2

    target_file=CACHE_DIR/"gierjatowicz_heston_targets.csv"
    if target_file.exists():
        synth_contracts=pd.read_csv(target_file)
    else:
        rows=[]; strikes=np.round(np.arange(0.8,1.2001,0.02),2); mats=np.arange(2,13,2)/12
        for T in tqdm(mats,desc="Heston targets",unit="maturity"):
            for K in strikes:
                F=HESTON["S0"]*math.exp(HESTON["r"]*T); D=math.exp(-HESTON["r"]*T)
                rows.append({"S":1.0,"F":F,"D":D,"K":K,"T":T,"k":math.log(K/F),"target_cp":"C",
                             "target_mid":heston_call(K,T),"target_vega":1.0,"am_settlement":0})
        synth_contracts=pd.DataFrame(rows); synth_contracts["optionid"]=np.arange(len(synth_contracts)); synth_contracts.to_csv(target_file,index=False)

    def build_synthetic_context(contracts):
        c=contracts.copy().reset_index(drop=True); c["contract_id"]=np.arange(len(c)); c["vega_scale"]=1.0
        boundaries=np.sort(c["T"].unique()); grid_np=build_time_grid(boundaries)
        midx=nearest_grid_indices(grid_np,c["T"].to_numpy(float)); smap={round(float(T),12):i for i,T in enumerate(boundaries)}
        sidx=np.array([smap[round(float(T),12)] for T in c["T"]],dtype=int)
        F=np.exp(HESTON["r"]*grid_np); D=np.exp(-HESTON["r"]*grid_np)
        return CalibrationContext(
            date=pd.Timestamp("2000-01-01"),spot=1.0,contracts=c,maturity_boundaries=boundaries,grid_np=grid_np,
            grid=torch.tensor(grid_np,dtype=torch.float32,device=DEVICE),
            maturity_index=torch.tensor(midx,dtype=torch.long,device=DEVICE),stage_index=sidx,
            stage_tensor=torch.tensor(sidx,dtype=torch.long,device=DEVICE),
            K=torch.tensor(c["K"].to_numpy(float),dtype=torch.float32,device=DEVICE),
            F_T=torch.tensor(c["F"].to_numpy(float),dtype=torch.float32,device=DEVICE),
            D_T=torch.tensor(c["D"].to_numpy(float),dtype=torch.float32,device=DEVICE),
            target=torch.tensor(c["target_mid"].to_numpy(float),dtype=torch.float32,device=DEVICE),
            vega_scale=torch.ones(len(c),dtype=torch.float32,device=DEVICE),
            cp_sign=torch.ones(len(c),dtype=torch.float32,device=DEVICE),
            am=torch.zeros(len(c),dtype=torch.long,device=DEVICE),
            F_grid=torch.tensor(np.vstack([F,F]),dtype=torch.float32,device=DEVICE),
            D_grid=torch.tensor(np.vstack([D,D]),dtype=torch.float32,device=DEVICE),
            loss_mode="raw"
        )

    synth_ctx=build_synthetic_context(synth_contracts)
    SYNTH_CFG=TrainConfig(
        epochs=600,paths_per_step=4_096,steps_per_epoch=6,quotes_per_maturity=21,
        monitor_paths=16_384,eval_every_sde=10,top_k=3,rerank_paths=50_000,save_every=10,
        lr_milestones=(300,480),seed_offset=20_000)
    
    synth_model,synth_hedge,synth_history,synth_rerank=train_v6(
        synth_ctx,"maturity_heads",SYNTH_CFG,"synthetic_gierjatowicz",warm_start_v5=False)
    
    z=antithetic_normals(100_000,len(synth_ctx.grid_np)-1,SEED+29_000)
    sd=objective_diagnostics(synth_model,synth_ctx,z); del z; gc.collect()
    print(f"Synthetic final price RMSE={sd['raw_rmse']:.6g}; MSE={sd['raw_rmse']**2:.6g}; MAE={sd['raw_mae']:.6g}")
    pd.DataFrame([{ "RMSE":sd["raw_rmse"],"MSE":sd["raw_rmse"]**2,"MAE":sd["raw_mae"] }]).to_csv(
        CACHE_DIR/"synthetic_gierjatowicz_summary.csv",index=False)


Rerank maturity_heads: 100%|██████████| 3/3 [00:09<00:00,  3.32s/model]


Independent rerank selected epoch 479: objective=0.00169, $RMSE=0.0017
Synthetic final price RMSE=0.0015425; MSE=2.3793e-06; MAE=0.00133464


Heston targets: 100%|██████████| 6/6 [00:00<00:00, 16.52maturity/s]

v6 synthetic_gierjatowicz [maturity_heads]: 100%|██████████| 600/600 [1:10:52<00:00,  7.09s/epoch, phase=hedge, loss=8.58e-05, lr=4.0e-05]

Rerank maturity_heads: 100%|██████████| 3/3 [00:09<00:00,  3.32s/model]

Independent rerank selected epoch 479: objective=0.00169, $RMSE=0.0017

Synthetic final price RMSE=0.0015425; MSE=2.3793e-06; MAE=0.00133464

### v6 stopping point

Once the main held-out/next-day/delta checks pass and the two optional experiments have been run, Neural LV has done the work required for the project-level benchmark. v7 should add only the five-date pricing/hedging panel and three-seed robustness check, then freeze Neural LV and move to Neural SLV.
